In [ ]:
# -*- coding: utf-8 -*-
"""
完整过拟合诊断 + 真实Alpha自动估算（修复日期对齐问题）
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Union, Dict, Any, Callable
from itertools import combinations
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==================== 配置 ====================
BACKTEST_ID = 'ddc5280cb56be915652d273e09adf753'  # 请修改为你的回测ID

# ==================== 获取回测数据 ====================
gt = get_backtest(BACKTEST_ID)
data = gt.get_results()
df = pd.DataFrame(data)

# 确保索引为日期（归一化到00:00:00）
if not isinstance(df.index, pd.DatetimeIndex):
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
    else:
        from datetime import timedelta
        start_date = pd.Timestamp('2013-01-01')
        dates = pd.date_range(start=start_date, periods=len(df), freq='B')
        df.index = dates

# 归一化日期到00:00:00（去掉时间部分）
df.index = pd.to_datetime(df.index.date)

# 计算日收益率
df['daily_returns'] = (1 + df['returns']) / (1 + df['returns'].shift(1)) - 1
bt_ret = df['daily_returns'][1:]

# 获取换手率
turnover_series = None
try:
    records_data = gt.get_records()
    if records_data and len(records_data) > 0:
        # 转换为 DataFrame
        records_df = pd.DataFrame(records_data)
        if 'DAILY_TURNOVER' in records_df.columns:
            records_df['time'] = pd.to_datetime(records_df['time'])
            records_df.set_index('time', inplace=True)
            records_df.index = pd.to_datetime(records_df.index.date)
            common_dates = bt_ret.index.intersection(records_df.index)
            if len(common_dates) > 0:
                turnover_series = records_df.loc[common_dates, 'DAILY_TURNOVER'].values
                print(f"✅ 成功获取真实换手率序列，日均换手率 {np.mean(turnover_series):.4f}")
            else:
                print("⚠️ 换手率记录日期与收益率日期无交集，使用默认换手率")
        else:
            print("⚠️ 记录中无 DAILY_TURNOVER 列，使用默认换手率")
    else:
        print("⚠️ 无记录数据，使用默认换手率")
except Exception as e:
    print(f"⚠️ 无法获取自定义记录，使用默认换手率。错误信息: {e}")

# ==================== 辅助函数 ====================
def _check_sample_size(returns: np.ndarray, min_days: int = 500) -> None:
    if len(returns) < min_days:
        print(f"⚠️ 警告：样本量仅 {len(returns)} 天，少于建议的 {min_days} 天，部分检验可靠性降低")

def _safe_sharpe(returns: np.ndarray, annual_factor: float = np.sqrt(252)) -> float:
    if len(returns) == 0:
        return 0.0
    mu = np.mean(returns)
    sigma = np.std(returns, ddof=1)
    if sigma < 1e-8:
        return 0.0
    return mu / sigma * annual_factor

def _annual_return(returns: np.ndarray, periods: int = 252) -> float:
    if len(returns) == 0:
        return 0.0
    total = np.prod(1 + returns) - 1
    return (1 + total) ** (periods / len(returns)) - 1

def _max_drawdown(returns: np.ndarray) -> float:
    if len(returns) == 0:
        return 0.0
    cum = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cum)
    return np.min(cum / peak - 1)

def _calmar_ratio(returns: np.ndarray, periods: int = 252) -> float:
    ann_ret = _annual_return(returns, periods)
    mdd = abs(_max_drawdown(returns))
    return ann_ret / mdd if mdd > 0 else np.inf

# ==================== 标准过拟合检验函数 ====================
def check_temporal_stability(returns: np.ndarray, window: int = 252, step: int = 21) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if n < window:
        return {"is_stable": False, "message": "样本量小于窗口长度", "p_value": 1.0}
    rolling_sharpes = []
    for i in range(0, n - window + 1, step):
        seg = returns[i:i+window]
        rolling_sharpes.append(_safe_sharpe(seg))
    if len(rolling_sharpes) < 2:
        return {"is_stable": True, "message": "滚动窗口数量不足", "p_value": 1.0}
    mean_sharpe = np.mean(rolling_sharpes)
    std_sharpe = np.std(rolling_sharpes)
    cv = std_sharpe / abs(mean_sharpe) if mean_sharpe != 0 else np.inf
    n_groups = min(3, len(rolling_sharpes) // 2)
    if n_groups >= 2:
        group_size = len(rolling_sharpes) // n_groups
        groups = [rolling_sharpes[i*group_size:(i+1)*group_size] for i in range(n_groups-1)]
        groups.append(rolling_sharpes[(n_groups-1)*group_size:])
        _, p_value = stats.kruskal(*groups)
    else:
        p_value = 1.0
    stable = (cv < 1.0) and (p_value > 0.05)
    print(f"滚动夏普均值: {mean_sharpe:.2f}, 标准差: {std_sharpe:.2f}")
    print(f"变异系数: {cv:.2f} (越小越稳定)")
    print(f"Kruskal-Wallis p值: {p_value:.4f} (>0.05表示各时期表现无显著差异)")
    print(f"稳定性结论: {'✅ 稳定' if stable else '⚠️ 不稳定'}")
    return {"rolling_sharpes": rolling_sharpes, "mean_sharpe": mean_sharpe, "std_sharpe": std_sharpe, "cv": cv, "p_value": p_value, "is_stable": stable}

def runs_test(returns: np.ndarray, zero_treatment: str = "separate", random_state: int = 42) -> Dict[str, Any]:
    returns = np.asarray(returns)
    if zero_treatment == "separate":
        signs = np.where(returns > 0, 1, np.where(returns < 0, -1, 0))
    elif zero_treatment == "ignore":
        signs = np.sign(returns)
        signs = signs[signs != 0]
    elif zero_treatment == "sign":
        signs = np.sign(returns)
    else:
        raise ValueError("zero_treatment must be 'separate', 'ignore', or 'sign'")
    n = len(signs)
    if n == 0:
        return {"is_random": False, "message": "无有效数据", "p_value": 1.0}
    unique, counts = np.unique(signs, return_counts=True)
    counts_dict = dict(zip(unique, counts))
    n_pos = counts_dict.get(1, 0)
    n_neg = counts_dict.get(-1, 0)
    n_zero = counts_dict.get(0, 0)
    runs = 1
    for i in range(1, n):
        if signs[i] != signs[i-1]:
            runs += 1
    rng = np.random.RandomState(random_state)
    if zero_treatment == "separate" and n_zero > 0:
        n_perm = 2000
        runs_perm = []
        for _ in range(n_perm):
            perm = rng.permutation(signs)
            r = 1 + np.sum(perm[:-1] != perm[1:])
            runs_perm.append(r)
        runs_perm = np.array(runs_perm)
        p_value = np.mean(runs_perm >= runs) if runs >= np.mean(runs_perm) else np.mean(runs_perm <= runs)
        p_value = min(p_value, 1 - p_value) * 2
        expected_runs = np.mean(runs_perm)
        z_stat = (runs - expected_runs) / np.std(runs_perm)
    else:
        n1, n2 = n_pos, n_neg
        if n1 == 0 or n2 == 0:
            return {"is_random": True, "message": "序列全同号", "runs": 1, "p_value": 1.0}
        expected_runs = (2.0 * n1 * n2) / n + 1.0
        numerator = 2.0 * n1 * n2 * (2.0 * n1 * n2 - n)
        denominator = n * n * (n - 1.0)
        var_runs = numerator / denominator if denominator > 0 else 0.0
        std_runs = np.sqrt(var_runs)
        if std_runs > 0:
            z_stat = (runs - expected_runs) / std_runs
            p_value = 2.0 * (1.0 - stats.norm.cdf(abs(z_stat)))
        else:
            z_stat, p_value = 0.0, 1.0
    print(f"样本量: {n}, 正: {n_pos}, 负: {n_neg}, 零: {n_zero if zero_treatment=='separate' else 0}")
    print(f"实际游程数: {runs}, 期望游程数: {expected_runs:.2f}")
    print(f"Z统计量: {z_stat:.2f}, p值: {p_value:.4f}")
    return {"runs": runs, "expected_runs": expected_runs, "z_stat": z_stat, "p_value": p_value, "is_random": p_value > 0.05}

def variance_ratio_test(returns: np.ndarray, q_list: List[int] = [2, 5, 10]) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    mu = np.mean(returns)
    y = returns - mu
    y2 = y ** 2
    sum_y2 = np.sum(y2)
    var_1 = sum_y2 / n
    results = {}
    for q in q_list:
        if n <= q:
            continue
        y_q = np.array([np.sum(y[i:i+q]) for i in range(n - q + 1)])
        var_q = np.sum(y_q**2) / (n - q + 1) / q
        vr = var_q / var_1 if var_1 > 0 else 1.0
        delta = np.zeros(q-1)
        for j in range(1, q):
            w = 2 * (q - j) / q
            num = np.dot(y2[j:], y2[:n-j])
            denom = sum_y2 ** 2
            delta[j-1] = w * (num / denom) if denom > 0 else 0.0
        se = np.sqrt(2 * np.sum(delta))
        z_stat = (vr - 1) / se if se > 0 else 0.0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        results[q] = {"vr": vr, "z_stat": z_stat, "p_value": p_value, "is_random_walk": p_value > 0.05}
        print(f"q={q}: VR={vr:.4f}, Z={z_stat:.2f}, p={p_value:.4f} -> {'随机游走' if p_value>0.05 else '非随机'}")
    p_values = [res["p_value"] for res in results.values()]
    min_p = min(p_values) if p_values else 1.0
    return {"by_q": results, "min_p_value": min_p, "is_random_walk": min_p > 0.05}

def chow_test(returns: np.ndarray, break_point: Optional[int] = None) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if break_point is None:
        break_point = n // 2
    if break_point < 2 or break_point > n-2:
        return {"has_break": False, "message": "断点位置无效", "p_value": 1.0}
    y = returns.reshape(-1, 1)
    X = np.ones((n, 1))
    beta_full = np.linalg.lstsq(X, y, rcond=None)[0]
    resid_full = y - X @ beta_full
    rss_full = np.sum(resid_full**2)
    X1, y1 = X[:break_point], y[:break_point]
    beta1 = np.linalg.lstsq(X1, y1, rcond=None)[0]
    rss1 = np.sum((y1 - X1 @ beta1)**2)
    X2, y2 = X[break_point:], y[break_point:]
    beta2 = np.linalg.lstsq(X2, y2, rcond=None)[0]
    rss2 = np.sum((y2 - X2 @ beta2)**2)
    k = 1
    numerator = (rss_full - (rss1 + rss2)) / k
    denominator = (rss1 + rss2) / (n - 2*k)
    f_stat = numerator / denominator if denominator > 0 else 0.0
    p_value = 1 - stats.f.cdf(f_stat, k, n - 2*k)
    has_break = p_value < 0.05
    print(f"断点位置: {break_point}, 前段夏普: {_safe_sharpe(returns[:break_point]):.2f}, 后段夏普: {_safe_sharpe(returns[break_point:]):.2f}")
    print(f"Chow F={f_stat:.3f}, p={p_value:.4f} -> {'存在断点' if has_break else '无明显断点'}")
    return {"f_stat": f_stat, "p_value": p_value, "has_break": has_break}

def block_bootstrap_sensitivity(returns: np.ndarray, n_bootstrap: int = 1000, block_size: Optional[int] = None, random_state: int = 42) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if block_size is None:
        block_size = int(round(n ** (1/3)))
    block_size = max(1, min(block_size, n // 2))
    rng = np.random.RandomState(random_state)
    original_sharpe = _safe_sharpe(returns)
    bootstrap_sharpes = []
    for _ in range(n_bootstrap):
        n_blocks = int(np.ceil(n / block_size))
        blocks = []
        for _ in range(n_blocks):
            start = rng.randint(0, n - block_size + 1)
            blocks.append(returns[start:start+block_size])
        sample = np.concatenate(blocks)[:n]
        bootstrap_sharpes.append(_safe_sharpe(sample))
    bootstrap_sharpes = np.array(bootstrap_sharpes)
    percentile = stats.percentileofscore(bootstrap_sharpes, original_sharpe)
    is_extreme = (percentile < 2.5) or (percentile > 97.5)
    print(f"原始夏普: {original_sharpe:.2f}, Bootstrap中位数: {np.median(bootstrap_sharpes):.2f}, 百分位: {percentile:.1f}%")
    print(f"结论: {'极端' if is_extreme else '未发现极端异常'}")
    return {"original_sharpe": original_sharpe, "bootstrap_sharpes": bootstrap_sharpes, "percentile": percentile, "is_extreme": is_extreme}

def rolling_cross_validation(returns: np.ndarray, train_window: int = 504, test_window: int = 63, step: int = 21, strategy_func: Optional[Callable[[np.ndarray], np.ndarray]] = None) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if n < train_window + test_window:
        return {"message": "样本量不足", "is_overfit": False, "decay": 0.0}
    oos_sharpes = []
    for start in range(0, n - train_window - test_window + 1, step):
        train_end = start + train_window
        test_end = train_end + test_window
        train_returns = returns[start:train_end]
        test_returns = returns[train_end:test_end]
        if strategy_func is not None:
            try:
                pred_returns = strategy_func(train_returns)
                if len(pred_returns) != test_window:
                    pred_returns = pred_returns[:test_window]
                oos_sharpe = _safe_sharpe(pred_returns)
            except Exception:
                continue
        else:
            oos_sharpe = _safe_sharpe(test_returns)
        oos_sharpes.append(oos_sharpe)
    if len(oos_sharpes) == 0:
        return {"message": "无有效窗口", "is_overfit": False, "decay": 0.0}
    mean_oos = np.mean(oos_sharpes)
    std_oos = np.std(oos_sharpes)
    full_sharpe = _safe_sharpe(returns)
    decay = full_sharpe - mean_oos
    print(f"全样本夏普: {full_sharpe:.2f}")
    print(f"样本外夏普均值: {mean_oos:.2f} ± {std_oos:.2f}")
    print(f"样本外衰减: {decay:.2f}")
    return {"oos_sharpes": oos_sharpes, "mean_oos": mean_oos, "std_oos": std_oos, "decay": decay, "is_overfit": decay > 0.5}

def approximate_pbo(returns: np.ndarray, n_trials: int = 100, block_size: Optional[int] = None, random_state: int = 42) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if block_size is None:
        block_size = int(round(n ** (1/3)))
    block_size = max(1, min(block_size, n // 4))
    rng = np.random.RandomState(random_state)
    trials_matrix = np.zeros((n, n_trials))
    for i in range(n_trials):
        n_blocks = int(np.ceil(n / block_size))
        blocks = []
        for _ in range(n_blocks):
            start = rng.randint(0, n - block_size + 1)
            blocks.append(returns[start:start+block_size])
        sample = np.concatenate(blocks)[:n]
        trials_matrix[:, i] = sample
    sharpes = np.array([_safe_sharpe(trials_matrix[:, i]) for i in range(n_trials)])
    valid = np.isfinite(sharpes)
    sharpes = sharpes[valid]
    trials_matrix = trials_matrix[:, valid]
    n_trials_valid = trials_matrix.shape[1]
    if n_trials_valid < 10:
        return {"pbo": np.nan, "message": "有效变体数量不足"}
    S = min(8, n // 4)
    if S % 2 != 0:
        S = max(2, S - 1)
    block_len = n // S
    combos = list(combinations(range(S), S//2))
    oos_ranks = []
    for combo in combos:
        is_mask = np.zeros(S, dtype=bool)
        is_mask[list(combo)] = True
        oos_mask = ~is_mask
        is_idx = np.concatenate([np.arange(i*block_len, (i+1)*block_len) for i, m in enumerate(is_mask) if m])
        oos_idx = np.concatenate([np.arange(i*block_len, (i+1)*block_len) for i, m in enumerate(oos_mask) if m])
        is_perf = np.array([_safe_sharpe(trials_matrix[is_idx, j]) for j in range(n_trials_valid)])
        oos_perf = np.array([_safe_sharpe(trials_matrix[oos_idx, j]) for j in range(n_trials_valid)])
        valid_both = np.isfinite(is_perf) & np.isfinite(oos_perf)
        if np.sum(valid_both) < 5:
            continue
        is_perf = is_perf[valid_both]
        oos_perf = oos_perf[valid_both]
        best_idx = np.argmax(is_perf)
        oos_rank = np.mean(oos_perf >= oos_perf[best_idx])
        oos_ranks.append(oos_rank)
    if len(oos_ranks) == 0:
        return {"pbo": np.nan, "message": "无有效组合"}
    oos_ranks = np.array(oos_ranks)
    pbo = np.mean(oos_ranks <= 0.5)
    print(f"近似PBO: {pbo:.3f} ({'低' if pbo<0.2 else '中' if pbo<0.4 else '高'}风险)")
    print("⚠️ 注意：此PBO为近似值，可能低估真实过拟合风险。完整评估需提供策略参数搜索矩阵。")
    return {"pbo": pbo, "oos_ranks": oos_ranks, "warning": "近似PBO，可能低估真实过拟合风险"}

def live_vs_backtest_comparison(bt_returns: np.ndarray, live_returns: np.ndarray) -> Dict[str, Any]:
    bt = np.asarray(bt_returns)
    live = np.asarray(live_returns)
    bt_sharpe = _safe_sharpe(bt)
    live_sharpe = _safe_sharpe(live)
    bt_ret = _annual_return(bt)
    live_ret = _annual_return(live)
    sharpe_decay = bt_sharpe - live_sharpe
    ks_stat, ks_p = stats.ks_2samp(bt, live)
    print(f"回测夏普: {bt_sharpe:.2f}, 实盘夏普: {live_sharpe:.2f}, 衰减: {sharpe_decay:.2f}")
    print(f"回测年化收益: {bt_ret:.2%}, 实盘年化收益: {live_ret:.2%}")
    print(f"KS检验 p值: {ks_p:.4f} (分布一致性)")
    is_consistent = (ks_p > 0.05) and (sharpe_decay < 1.0)
    return {"bt_sharpe": bt_sharpe, "live_sharpe": live_sharpe, "sharpe_decay": sharpe_decay, "bt_ret": bt_ret, "live_ret": live_ret, "ks_p": ks_p, "is_consistent": is_consistent}

def _plot_diagnostics(returns: np.ndarray, results: Dict[str, Any], live_returns: Optional[np.ndarray] = None) -> None:
    n_plots = 6 if live_returns is not None else 5
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten()
    ax = axes[0]
    cum = np.cumprod(1 + returns)
    ax.plot(cum, label="回测", color="blue")
    if live_returns is not None:
        live_cum = np.cumprod(1 + live_returns)
        ax.plot(np.arange(len(returns), len(returns)+len(live_returns)), live_cum * cum[-1], label="实盘", color="red", linestyle="--")
    ax.set_title("累计收益曲线")
    ax.legend()
    ax = axes[1]
    if "rolling_sharpes" in results["stability"]:
        rs = results["stability"]["rolling_sharpes"]
        ax.plot(rs, color="green")
        ax.axhline(y=0, linestyle="--", color="gray")
        ax.set_title("滚动窗口夏普比率")
        ax.set_xlabel("窗口序号")
    ax = axes[2]
    cost_data = results["cost"]
    ax.plot(cost_data["cost_bps"], cost_data["net_sharpes"], marker="o", color="red")
    ax.axhline(y=0, linestyle="--", color="gray")
    ax.set_title("夏普比率 vs 交易成本")
    ax.set_xlabel("成本 (bps)")
    ax = axes[3]
    if "oos_sharpes" in results["cv"] and len(results["cv"]["oos_sharpes"]) > 0:
        ax.hist(results["cv"]["oos_sharpes"], bins=15, color="purple", alpha=0.7)
        ax.axvline(x=results["cv"]["mean_oos"], color="black", linestyle="--", label="均值")
        ax.set_title("样本外夏普分布")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "样本外数据不足", ha="center", va="center")
        ax.set_title("样本外夏普分布")
    ax = axes[4]
    if "bootstrap_sharpes" in results["bootstrap"]:
        ax.hist(results["bootstrap"]["bootstrap_sharpes"], bins=30, color="orange", alpha=0.7)
        ax.axvline(x=results["bootstrap"]["original_sharpe"], color="red", linestyle="--", label="原始")
        ax.set_title("Bootstrap夏普分布")
        ax.legend()
    if live_returns is not None and n_plots >= 6:
        ax = axes[5]
        metrics = ["夏普", "年化收益", "最大回撤"]
        bt_vals = [results["live_compare"]["bt_sharpe"], results["live_compare"]["bt_ret"], _max_drawdown(returns)]
        live_vals = [results["live_compare"]["live_sharpe"], results["live_compare"]["live_ret"], _max_drawdown(live_returns)]
        x = np.arange(len(metrics))
        width = 0.35
        ax.bar(x - width/2, bt_vals, width, label="回测", color="blue")
        ax.bar(x + width/2, live_vals, width, label="实盘", color="red")
        ax.set_xticks(x)
        ax.set_xticklabels(metrics)
        ax.set_title("回测 vs 实盘关键指标")
        ax.legend()
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    plt.show()

# ==================== 新增检测函数 ====================
def monte_carlo_synthetic_test(returns, n_simulations=1000, random_state=42):
    returns = np.asarray(returns).flatten()
    rng = np.random.RandomState(random_state)
    orig_sharpe = _safe_sharpe(returns)
    sims = []
    for _ in range(n_simulations):
        shuffled = rng.permutation(returns)
        sims.append(_safe_sharpe(shuffled))
    sims = np.array(sims)
    pct = stats.percentileofscore(sims, orig_sharpe)
    print(f"蒙特卡洛合成检验 (完全打乱): 原始夏普 {orig_sharpe:.2f}，随机序列夏普均值 {np.mean(sims):.2f} ± {np.std(sims):.2f}")
    print(f"原始夏普百分位: {pct:.1f}% {'(显著高于随机)' if pct > 95 else '(未显著高于随机)'}")
    return {"original_sharpe": orig_sharpe, "sim_sharpes": sims, "percentile": pct, "is_significant": pct > 95}

def enhanced_transaction_cost_sensitivity(returns: np.ndarray, turnover_series: Optional[np.ndarray] = None, cost_bps_range: Tuple[float, float] = (0, 50), n_points: int = 11) -> Dict[str, Any]:
    returns = np.asarray(returns)
    if turnover_series is not None:
        turnover_series = np.asarray(turnover_series)
        if len(turnover_series) != len(returns):
            print("⚠️ 换手率序列长度与收益序列不一致，将使用默认换手率")
            turnover_series = None
    if turnover_series is None:
        print("⚠️ 未提供换手率，使用默认日均0.03")
        avg_turnover = 0.03
        turnover_series = np.full_like(returns, avg_turnover)
    else:
        avg_turnover = np.mean(turnover_series)
        print(f"使用真实换手率，日均 {avg_turnover:.4f}")
    cost_bps = np.linspace(cost_bps_range[0], cost_bps_range[1], n_points)
    net_sharpes = []
    original_sharpe = _safe_sharpe(returns)
    for c_bps in cost_bps:
        cost_per_trade = c_bps / 10000.0
        daily_cost = turnover_series * 2 * cost_per_trade
        net_returns = returns - daily_cost
        net_sharpes.append(_safe_sharpe(net_returns))
    idx_10bps = np.argmin(np.abs(cost_bps - 10))
    sharpe_10bps = net_sharpes[idx_10bps]
    is_robust = (sharpe_10bps > 0) and (sharpe_10bps > original_sharpe * 0.5)
    print(f"原始夏普: {original_sharpe:.2f}, 10bps后: {sharpe_10bps:.2f}, 50bps后: {net_sharpes[-1]:.2f}")
    print(f"成本鲁棒性: {'✅ 良好' if is_robust else '⚠️ 敏感'}")
    return {"cost_bps": cost_bps, "net_sharpes": net_sharpes, "is_cost_robust": is_robust, "sharpe_10bps": sharpe_10bps, "avg_turnover": avg_turnover}

def risk_ratio_stability(returns: np.ndarray, window: int = 252, step: int = 63) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if n < window:
        return {"message": "样本量小于窗口", "calmar_mean": np.nan, "calmar_std": np.nan}
    calmar_list = []
    for i in range(0, n - window + 1, step):
        seg = returns[i:i+window]
        calmar_list.append(_calmar_ratio(seg))
    calmar_arr = np.array(calmar_list)
    mean_calmar = np.mean(calmar_arr)
    std_calmar = np.std(calmar_arr)
    cv_calmar = std_calmar / abs(mean_calmar) if mean_calmar != 0 else np.inf
    print(f"滚动Calmar比率 (窗口{window}天): 均值 {mean_calmar:.2f}, 标准差 {std_calmar:.2f}, 变异系数 {cv_calmar:.2f}")
    return {"calmar_rolling": calmar_arr, "mean_calmar": mean_calmar, "std_calmar": std_calmar, "cv_calmar": cv_calmar}

def var_backtest(returns: np.ndarray, var_level: float = 0.05, window: int = 252) -> Dict[str, Any]:
    returns = np.asarray(returns)
    n = len(returns)
    if n < window + 1:
        return {"error": "样本量不足"}
    var_series = []
    violations = 0
    for i in range(window, n):
        hist_rets = returns[i-window:i]
        var = -np.percentile(hist_rets, var_level * 100)
        var_series.append(var)
        if -returns[i] > var:
            violations += 1
    failure_rate = violations / (n - window)
    print(f"VaR (95%) 回测: 失败率 {failure_rate:.2%}, 理论期望 {var_level:.2%}")
    if failure_rate > var_level * 2:
        print("⚠️ 失败率显著高于预期，策略尾部风险被低估")
    elif failure_rate < var_level / 2:
        print("⚠️ 失败率显著低于预期，可能过于保守")
    else:
        print("✅ 失败率与预期接近")
    return {"failure_rate": failure_rate, "expected_rate": var_level, "mean_var": np.mean(var_series), "violations": violations, "total_tests": n - window}

def sharpe_ci_bootstrap(returns: np.ndarray, n_bootstrap: int = 10000, conf_level: float = 0.95, random_state: int = 42) -> Dict[str, Any]:
    returns = np.asarray(returns)
    rng = np.random.RandomState(random_state)
    original_sharpe = _safe_sharpe(returns)
    boot_sharpes = []
    n = len(returns)
    for _ in range(n_bootstrap):
        sample = returns[rng.randint(0, n, n)]
        boot_sharpes.append(_safe_sharpe(sample))
    boot_sharpes = np.array(boot_sharpes)
    lower = np.percentile(boot_sharpes, (1 - conf_level) / 2 * 100)
    upper = np.percentile(boot_sharpes, (1 + conf_level) / 2 * 100)
    print(f"夏普比率 {conf_level:.0%} 自助置信区间: [{lower:.2f}, {upper:.2f}] (原始 {original_sharpe:.2f})")
    return {"original_sharpe": original_sharpe, "ci_lower": lower, "ci_upper": upper, "conf_level": conf_level, "bootstrap_sharpes": boot_sharpes}

# ==================== 真实Alpha估算函数 ====================
def estimate_real_alpha(returns: pd.Series, benchmark_code: str = '000300.XSHG', window_years: int = 3, step_days: int = 252) -> float:
    """
    根据回测收益率序列自动估算真实Alpha，并评估可靠性
    """
    # 确保索引是 datetime 类型（归一化到日期）
    if not isinstance(returns.index, pd.DatetimeIndex):
        print("收益率索引不是日期类型，无法估算")
        return None
    
    # 计算回测年限
    total_days = len(returns)
    total_years = total_days / 252
    print(f"\n回测数据年限: {total_years:.1f}年 (共{total_days}个交易日)")
    
    # 检查最低年限要求
    if total_years < window_years:
        print(f"❌ 回测年限不足{window_years}年，无法进行滚动窗口验证。建议回测至少{window_years}年。")
        return None
    
    # 归一化日期
    returns.index = pd.to_datetime(returns.index.date)
    
    # 获取基准指数
    start_str = returns.index[0].strftime('%Y-%m-%d')
    end_str = returns.index[-1].strftime('%Y-%m-%d')
    try:
        bench_df = get_price(benchmark_code, start_date=start_str, end_date=end_str,
                             fields=['close'], fq='pre', panel=False)
        bench_df['returns'] = bench_df['close'].pct_change()
        bench_ret = bench_df['returns'].iloc[1:]
        bench_ret.index = pd.to_datetime(bench_ret.index.date)
    except Exception as e:
        print(f"无法获取基准指数 {benchmark_code}: {e}")
        return None
    
    # 对齐日期
    common = returns.index.intersection(bench_ret.index)
    if len(common) < 100:
        print(f"对齐后交易日数 {len(common)} 不足100，无法估算")
        return None
    
    excess = returns.loc[common] - bench_ret.loc[common]
    
    # 滚动窗口
    window_days = window_years * 252
    n = len(excess)
    
    ann_excess_list = []
    for start in range(0, n - window_days + 1, step_days):
        win = excess.iloc[start:start+window_days]
        total = np.prod(1 + win) - 1
        years = len(win) / 252
        ann = (1 + total) ** (1 / years) - 1 if years > 0 else 0
        ann_excess_list.append(ann * 100)
    
    num_windows = len(ann_excess_list)
    print(f"生成{window_years}年窗口数量: {num_windows}个 (步长{step_days//252}年)")
    
    # 可靠性评级
    if num_windows == 0:
        print("❌ 没有完整窗口，无法估算")
        return None
    elif num_windows <= 2:
        reliability = "🔴 极低"
        advice = f"回测年限仅{total_years:.1f}年，窗口数{num_windows}个，估算结果偶然性极大。建议将回测区间延长至至少8年以上。"
    elif num_windows <= 4:
        reliability = "🟡 中等"
        advice = f"回测年限{total_years:.1f}年，窗口数{num_windows}个，估算结果有一定参考价值，但不稳定。建议延长回测至10年以上。"
    elif num_windows <= 7:
        reliability = "🟢 较高"
        advice = f"回测年限{total_years:.1f}年，窗口数{num_windows}个，估算结果相对可靠。"
    else:
        reliability = "✅ 高"
        advice = f"回测年限{total_years:.1f}年，窗口数{num_windows}个，统计意义较强，可信度高。"
    
    real_alpha = np.median(ann_excess_list)
    total_excess = np.prod(1 + excess) - 1
    years_total = len(excess) / 252
    ann_total_excess = (1 + total_excess) ** (1 / years_total) - 1 if years_total > 0 else 0
    
    print(f"\n{'='*60}")
    print("真实Alpha估算（滚动窗口法）")
    print(f"基准指数: {benchmark_code}")
    print(f"窗口长度: {window_years}年，步长: {step_days//252}年，共{num_windows}个窗口")
    print(f"各窗口年化超额收益: {[f'{x:.1f}%' for x in ann_excess_list]}")
    print(f"中位数（真实Alpha）: {real_alpha:.2f}%")
    print(f"平均值: {np.mean(ann_excess_list):.2f}% ± {np.std(ann_excess_list):.2f}%")
    print(f"正向窗口比例: {sum(1 for x in ann_excess_list if x > 0)}/{num_windows}")
    print(f"\n全历史年化超额收益（回测CAPM Alpha）: {ann_total_excess*100:.2f}%")
    print(f"虚假收益 ≈ {ann_total_excess*100 - real_alpha:.2f}%")
    print(f"\n可靠性评级: {reliability}")
    print(f"建议: {advice}")
    print("="*60)
    
    return real_alpha

# ==================== 综合诊断函数 ====================
def comprehensive_overfitting_check(
    returns: pd.Series,
    live_returns: Optional[np.ndarray] = None,
    strategy_func: Optional[Callable[[np.ndarray], np.ndarray]] = None,
    strategy_name: str = "Strategy",
    show_plots: bool = True,
    bonferroni_alpha: float = 0.05,
    random_state: int = 42,
    turnover_series: Optional[np.ndarray] = None,
    benchmark_code: str = '000300.XSHG'
) -> Dict[str, Any]:
    returns_arr = np.asarray(returns)
    _check_sample_size(returns_arr)
    print(f"\n{'='*60}")
    print(f"过拟合诊断报告: {strategy_name}")
    print(f"回测样本: {len(returns_arr)} 天 ({len(returns_arr)/252:.1f} 年)")
    if live_returns is not None:
        print(f"实盘样本: {len(live_returns)} 天")
    print(f"{'='*60}")
    ann_ret = _annual_return(returns_arr)
    vol = np.std(returns_arr) * np.sqrt(252)
    sharpe = ann_ret / vol if vol > 0 else 0
    max_dd = _max_drawdown(returns_arr)
    print(f"\n基础指标:")
    print(f"  年化收益: {ann_ret:.2%}")
    print(f"  年化波动: {vol:.2%}")
    print(f"  夏普比率: {sharpe:.2f}")
    print(f"  最大回撤: {max_dd:.2%}")

    results = {}
    p_values = []

    print(f"\n--- 1. 滚动夏普稳定性 ---")
    results["stability"] = check_temporal_stability(returns_arr)
    p_values.append(results["stability"]["p_value"])

    print(f"\n--- 2. 游程随机性检验 ---")
    results["runs"] = runs_test(returns_arr, random_state=random_state)
    p_values.append(results["runs"]["p_value"])

    print(f"\n--- 3. 方差比率检验 ---")
    results["vr"] = variance_ratio_test(returns_arr)
    p_values.append(results["vr"]["min_p_value"])

    print(f"\n--- 4. 结构断点检验 ---")
    results["chow"] = chow_test(returns_arr)
    p_values.append(results["chow"]["p_value"])

    print(f"\n--- 5. 块自助法敏感性 ---")
    results["bootstrap"] = block_bootstrap_sensitivity(returns_arr, random_state=random_state)

    print(f"\n--- 6. 交易成本敏感性 (增强版) ---")
    results["cost"] = enhanced_transaction_cost_sensitivity(returns_arr, turnover_series=turnover_series)

    print(f"\n--- 7. 滚动样本外验证 ---")
    results["cv"] = rolling_cross_validation(returns_arr, strategy_func=strategy_func)

    print(f"\n--- 8. 近似PBO计算 ---")
    results["approx_pbo"] = approximate_pbo(returns_arr, random_state=random_state)

    print(f"\n--- 11. 风险比率稳定性 (Calmar) ---")
    results["risk_ratio_stability"] = risk_ratio_stability(returns_arr)

    print(f"\n--- 12. VaR回测 (历史模拟) ---")
    results["var_backtest"] = var_backtest(returns_arr)

    print(f"\n--- 13. 夏普比率自助置信区间 ---")
    results["sharpe_ci"] = sharpe_ci_bootstrap(returns_arr, random_state=random_state)

    if live_returns is not None:
        print(f"\n--- 14. 实盘 vs 回测对比 ---")
        results["live_compare"] = live_vs_backtest_comparison(returns_arr, live_returns)

    n_tests = len(p_values)
    bonferroni_threshold = bonferroni_alpha / n_tests if n_tests > 0 else bonferroni_alpha
    print(f"\n多重检验校正: {n_tests} 项统计检验, Bonferroni阈值 = {bonferroni_threshold:.4f}")

    statistical_risks = []
    effect_risks = []
    if results["stability"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("时间不稳定")
    if results["runs"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("收益序列非随机")
    if results["vr"]["min_p_value"] < bonferroni_threshold:
        statistical_risks.append("非随机游走")
    if results["chow"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("存在结构断点")
    if results["bootstrap"]["is_extreme"]:
        effect_risks.append("夏普对历史路径敏感")
    if not results["cost"]["is_cost_robust"]:
        effect_risks.append("交易成本敏感")
    if results["cv"].get("is_overfit", False):
        effect_risks.append("样本外衰减过大")
    if live_returns is not None and not results.get("live_compare", {}).get("is_consistent", True):
        effect_risks.append("实盘与回测显著偏离")

    results["statistical_risks"] = statistical_risks
    results["effect_risks"] = effect_risks

    if show_plots:
        _plot_diagnostics(returns_arr, results, live_returns)

    return results

# ==================== 主程序 ====================
if __name__ == "__main__":
    # 运行标准过拟合诊断
    live_ret = None
    result = comprehensive_overfitting_check(bt_ret, live_returns=live_ret, 
                                             strategy_name="MyStrategy", show_plots=True,turnover_series=turnover_series)
    
    # ==================== 策略关键指标汇总 ====================
    ann_ret = _annual_return(bt_ret)
    vol = np.std(bt_ret) * np.sqrt(252)
    sharpe = ann_ret / vol if vol > 0 else 0
    max_dd = _max_drawdown(bt_ret)
    
    print("\n" + "="*60)
    print("策略关键指标汇总")
    print("="*60)
    
    def print_aligned(label, value, comment):
        line = f"{label}: {value}"
        padding = max(1, 50 - len(line))
        print(line + " " * padding + comment)
    
    print_aligned("年化收益", f"{ann_ret:.2%}", "✅ 优秀")
    print_aligned("年化波动", f"{vol:.2%}", "⚠️ 中等偏高")
    print_aligned("夏普比率", f"{sharpe:.2f}", "✅ 优秀")
    print_aligned("最大回撤", f"{max_dd:.2%}", "✅ 良好")
    
    stab = result.get('stability', {})
    p_stab = stab.get('p_value', 1)
    is_stable = stab.get('is_stable', False)
    print_aligned("时间稳定性 p值", f"{p_stab:.4f} ({'稳定' if is_stable else '不稳定'})", 
                  "🔴 极显著不稳定" if p_stab < 0.05 and not is_stable else "✅ 稳定")
    
    runs = result.get('runs', {})
    p_runs = runs.get('p_value', 1)
    is_random = runs.get('is_random', False)
    print_aligned("游程检验 p值", f"{p_runs:.4f} ({'随机' if is_random else '非随机'})",
                  "🔴 非随机（趋势策略特性，但结合其他指标有过度拟合嫌疑）")
    
    vr = result.get('vr', {})
    p_vr = vr.get('min_p_value', 1)
    print_aligned("方差比率最小p值", f"{p_vr:.4f}", "⚠️ 5日尺度拒绝随机游走（短期可预测性）")
    
    chow = result.get('chow', {})
    p_chow = chow.get('p_value', 1)
    has_break = chow.get('has_break', False)
    print_aligned("结构断点 p值", f"{p_chow:.4f} ({'存在断点' if has_break else '无断点'})", "✅ 良好")
    
    boot = result.get('bootstrap', {})
    percentile = boot.get('percentile', 50)
    is_extreme = boot.get('is_extreme', False)
    print_aligned("自助法夏普百分位", f"{percentile:.1f}% ({'极端' if is_extreme else '正常'})", "✅ 正常")
    
    cost = result.get('cost', {})
    sharpe_10bps = cost.get('sharpe_10bps', 0)
    is_robust = cost.get('is_cost_robust', False)
    print_aligned("成本后夏普 (10bps)", f"{sharpe_10bps:.2f} (鲁棒: {'是' if is_robust else '否'})", "✅ 良好")
    
    cv = result.get('cv', {})
    mean_oos = cv.get('mean_oos', 0)
    std_oos = cv.get('std_oos', 0)
    decay = cv.get('decay', 0)
    print_aligned("样本外夏普均值", f"{mean_oos:.2f} ± {std_oos:.2f} (衰减: {decay:.2f})", "⚠️ 均值尚可但标准差极大，极其不稳定")
    
    pbo = result.get('approx_pbo', {})
    pbo_val = pbo.get('pbo', 0)
    risk_level = '高' if pbo_val >= 0.4 else '中' if pbo_val >= 0.2 else '低'
    print_aligned("近似PBO", f"{pbo_val:.3f} ({risk_level}风险)", "🔴 高风险过拟合")
    
    var_test = result.get('var_backtest', {})
    if 'failure_rate' in var_test:
        fail_rate = var_test.get('failure_rate', 0)
        exp_rate = var_test.get('expected_rate', 0)
        print_aligned("VaR失败率", f"{fail_rate:.2%} (期望: {exp_rate:.2%})", "✅ 与预期接近")
    
    ci = result.get('sharpe_ci', {})
    ci_lower = ci.get('ci_lower', 0)
    ci_upper = ci.get('ci_upper', 0)
    print_aligned("夏普95%置信区间", f"[{ci_lower:.2f}, {ci_upper:.2f}]", "⚠️ 区间较宽，估计精度有限")
    
    calmar = result.get('risk_ratio_stability', {})
    mean_calmar = calmar.get('mean_calmar', 0)
    std_calmar = calmar.get('std_calmar', 0)
    print_aligned("滚动Calmar均值", f"{mean_calmar:.2f} ± {std_calmar:.2f}", "⚠️ 波动较大，稳定性一般")
    
    # ==================== 综合评估 ====================
    print(f"\n{'='*60}")
    print("综合评估:")
    stat_risks = result.get('statistical_risks', [])
    eff_risks = result.get('effect_risks', [])
    all_risks = stat_risks + eff_risks
    if all_risks:
        print(f"⚠️ 发现 {len(all_risks)} 项风险信号：")
        if stat_risks:
            print(f"  - 统计显著性风险 ({len(stat_risks)}项): {', '.join(stat_risks)}")
        if eff_risks:
            print(f"  - 效应量/启发式风险 ({len(eff_risks)}项): {', '.join(eff_risks)}")
        print("建议：简化策略、增加正则化、延长样本外测试、降低参数维度")
    else:
        print("✅ 未检测到明显过拟合特征，但仍建议进行独立样本外验证")
    print(f"{'='*60}")
    
    # ==================== 真实Alpha估算 ====================
print("\n正在估算真实Alpha...")
real_alpha_result = estimate_real_alpha(bt_ret, benchmark_code='000300.XSHG', window_years=3, step_days=252)

# ==================== 公式计算与直观对比 ====================
print("\n" + "="*60)
print("公式计算与直观对比")
print("="*60)

# 从之前计算的指标中获取
# 注意：需要确保这些变量在作用域内（它们已在主程序前面计算过）
# 如果变量不存在，可以从 result 中重新计算或直接使用已计算的 ann_ret 等

# 获取基准年化收益（最近一次真实Alpha计算中用的基准指数）
try:
    # 重新获取基准收益率用于计算CAPM Alpha（如果前面没保存）
    start_str = bt_ret.index[0].strftime('%Y-%m-%d')
    end_str = bt_ret.index[-1].strftime('%Y-%m-%d')
    bench_df = get_price('000300.XSHG', start_date=start_str, end_date=end_str,
                         fields=['close'], fq='pre', panel=False)
    bench_df['returns'] = bench_df['close'].pct_change()
    bench_ret_simple = bench_df['returns'].iloc[1:]
    bench_ret_simple.index = pd.to_datetime(bench_ret_simple.index.date)
    common_idx = bt_ret.index.intersection(bench_ret_simple.index)
    bench_annual = _annual_return(bench_ret_simple.loc[common_idx]) * 100
    strategy_annual = _annual_return(bt_ret.loc[common_idx]) * 100
    capm_alpha = strategy_annual - bench_annual
except:
    capm_alpha = 0

# 实盘损耗经验值（默认2.5%）
practical_loss = 2.5   # 交易成本+滑点+执行偏差+心理干扰

# 真实Alpha（如果估算成功）
if real_alpha_result is not None:
    real_alpha = real_alpha_result
else:
    real_alpha = 0

print("\n1. 回测超额收益（CAPM Alpha）")
print(f"   策略年化收益: {strategy_annual:.2f}%")
print(f"   基准年化收益: {bench_annual:.2f}%")
print(f"   → 回测超额 = {strategy_annual:.2f}% - {bench_annual:.2f}% = {capm_alpha:.2f}%")

print("\n2. 真实Alpha（滚动窗口法）")
print(f"   滚动窗口年化超额中位数 = {real_alpha:.2f}%")
print(f"   → 这就是更可信的真实Alpha估计值")

print("\n3. 虚假收益")
print(f"   虚假收益 = 回测超额 - 真实Alpha = {capm_alpha:.2f}% - {real_alpha:.2f}% = {capm_alpha - real_alpha:.2f}%")
if capm_alpha - real_alpha > 5:
    print("   ⚠️ 虚假收益较大，说明回测中大部分超额来自过拟合、幸存者偏差等")
else:
    print("   ✅ 虚假收益较小，说明回测相对可靠")

print("\n4. 实盘超额收益预测")
print(f"   实盘损耗（成本+滑点+执行+心理） ≈ {practical_loss}%")
print(f"   预测实盘超额 = 真实Alpha - 实盘损耗 = {real_alpha:.2f}% - {practical_loss:.2f}% = {real_alpha - practical_loss:.2f}%")

print("\n5. 实盘总收益预测")
print(f"   长期基准年化收益预期（沪深300）≈ 8%")
total_expected = 8 + (real_alpha - practical_loss)
print(f"   预测实盘总年化 ≈ 基准8% + 实盘超额({real_alpha - practical_loss:.2f}%) = {total_expected:.2f}%")

print("\n" + "="*60)
print("注意事项：")
print("• 预测值为历史估算，不保证未来实现。")
print("• 实盘损耗因人而异，请根据自身情况调整。")
print("• 建议用最低窗口值（保守）或中位数（中性）分别计算预期。")
print("="*60)